# Simple MCP demo
- Send a query to LLM, says it doesn't know
- Give it a tool to help, and it knows!<br />&nbsp;<br />

- MCP is like a USB standard for LLM/tool integration<br />&nbsp;<br />

- MCP has 3 components:
  - An MCP client which initiates a conversation and requests to the LLM and MCP server. MCP client is for example a chat client (Claude Desktop is great) or an IDE (Cursor, Windsurf) that is asking an LLM for help with code
  - An MCP server which provides tools for a purpose - A Fetch or Playwright tool to browse the Web, or a Context7 tool to look up Python module documentation in a vector DB
  - An LLM which supports tool use - any major LLM that follows the OpenAI tool use standard <br />&nbsp;<br />

- MCP Flow:
  - User launches MCP client, it connects to MCP server(s) per its configuration.
  - MCP client asks MCP server to list the tools it offers to the client.
  - MCP client can prompt the LLM, providing a list of available tools (calling signatures, and semantic descriptions of when to call them).
  - LLM responds to prompt. If, based on the prompt and the available tools, a tool would be the best way to answer the question, LLM will respond with a tool call request and the parameter values for the calling signature.
  - MCP client then calls the tools using the provided signature, adds the output from the tool to the conversation, and calls the LLM again with the updated conversation.
  - LLM may respond with further tool call requests, or provide a response.
  - That's mostly it. Besides executing tool calls, servers can also provide static resources for the client like docs, reference prompts, see the [docs on the home page](https://modelcontextprotocol.io/overview) <br />&nbsp;<br />

- &gt; 10,000 MCP servers available 
  - [MCP Market leaderboard (by GitHub stars)](https://mcpmarket.com/leaderboards)
  - [PulseMCP directory (by downloads)](https://www.pulsemcp.com/servers?sort=popular-30-days-desc)
  - [LobeHub](https://lobehub.com/mcp)
  - [Glama](https://glama.ai/mcp/servers)<br />&nbsp;<br />


- More info:
  - [Anthropic MCP Announcement](https://www.anthropic.com/news/model-context-protocol)
  - [Anthropic YouTube talk](https://www.youtube.com/watch?v=kQmXtrmQ5Zg)
  - [Model Context Protocol home page on GitHub](https://github.com/modelcontextprotocol)
  - [Composio intro](https://composio.dev/blog/what-is-model-context-protocol-mcp-explained)<br />&nbsp;<br />
      
      
  

![!image.png](q5AltSX5E3TfLsmtZp5jjLMU5U.png)


# Example Code

In [17]:
import sys
import os
import dotenv
import re
from datetime import datetime, timedelta
import time
from typing import Dict, Any, Optional, Annotated
from urllib.parse import urljoin, urlparse

import asyncio
import nest_asyncio

from contextlib import AsyncExitStack
import mcp
from mcp.client.stdio import stdio_client
from mcp import ClientSession, StdioServerParameters

import anthropic
from anthropic import Anthropic
import pdb


In [2]:
# load secrets from .env
dotenv.load_dotenv()

# enable asyncio in jupyter notebook
nest_asyncio.apply()

# Initialize plotly for Jupyter
# init_notebook_mode(connected=True)


In [3]:
client = anthropic.Anthropic()

# https://docs.anthropic.com/en/docs/about-claude/models/overview
claude_4_models = [
    "claude-opus-4-20250514",
    "claude-sonnet-4-20250514",
    "claude-3-5-haiku-20241022",
]

print("Available Claude models:")
print("\n".join(claude_4_models))
print()

# Try making a simple completion request to each:

message = "what is the airspeed velocity of an unladen swallow"
for model in claude_4_models:
    try:
        response = client.messages.create(
            model=model,
            max_tokens=200,
            messages=[{"role": "user", "content": message}]
        )
        print(f"✓ {model}")
        print(response.content[0].text)
        print()
    except Exception as e:
        print(f"✗ {model} - error: {str(e)}")

Available Claude models:
claude-opus-4-20250514
claude-sonnet-4-20250514
claude-3-5-haiku-20241022

✓ claude-opus-4-20250514
The airspeed velocity of an unladen swallow depends on whether you mean an African or European swallow!

This is, of course, a reference to the famous scene from "Monty Python and the Holy Grail." In reality:

- **European swallow**: Roughly 20.1 miles per hour (32.4 km/h)
- **African swallow**: Roughly 24 miles per hour (38.6 km/h)

These are estimates based on actual studies of swallow flight speeds. The European swallow estimate comes from a 2001 study by Dr. Anders Hedenström, though real flight speeds can vary based on wind conditions, whether the bird is migrating, foraging, or just cruising.

The Monty Python sketch has made this "fact" far more famous than it probably ever would have been otherwise!

✓ claude-sonnet-4-20250514
Ah, a classic Monty Python reference! 

The proper response is: "What do you mean? An African or European swallow?"

But if you wa

In [ ]:
"""
This swallow_server.py server implements a simple MCP server using FastMCP,
providing a tool unladen_swallow_airspeed, which returns a string based on input swallow_type
"""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("swallow-server")


@mcp.tool()
def unladen_swallow_airspeed(swallow_type: str) -> str:
    """Provides the airspeed velocity of an unladen swallow. Takes a 'swallow_type' argument ('african' or 'european')."""
    stype = swallow_type.strip().lower()
    if stype == 'african':
        return "31.1415926 km/h"
    elif stype == 'european':
        return "27.1828km/h"
    else:
        return "I don't know!"


def main():
    mcp.run()


if __name__ == "__main__":
    main()

## Test swallow_server.py using MCP Inspector
- `$ mcp dev swallow_server.py`
- click 'connect'
- click 'tools'
- click 'unladen_swallow_airspeed' tool
- enter parameters

![MCP Inspector Image](swallow_server.png)

In [11]:
MODEL = 'claude-sonnet-4-20250514'

class MCPClient:
    """An MCP client adapted to run in a Jupyter notebook.
    """
    def __init__(self):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.anthropic = Anthropic()
        self.tools = {}
        self.tools_reverse = {}

    def connect_to_server(self, server_script_path: str):
        """Connect to an MCP server and list its tools."""
        print(f"Connecting to server: {server_script_path}...")
        is_python = server_script_path.endswith('.py')
        if not is_python:
            raise ValueError("Server script must be a .py file")

        server_params = StdioServerParameters(
            command=sys.executable,  # Use the same python executable
            args=[server_script_path],
            env=None
        )

        pdb.set_trace()
        # print(server_params)
        response = asyncio.run(self.async_connect_to_server(server_params))
        # print(response)
        self.tools[server_script_path] = response.tools
        reverse_tool_dict = {tool.name: server_script_path for tool in response.tools}
        self.tools_reverse = {**self.tools_reverse, **reverse_tool_dict}
        print("\nConnection successful!")
        print("Available tools:", [tool.name for tool in self.tools[server_script_path]])

    async def async_connect_to_server(self, server_params):
        
        stdio_transport = await self.exit_stack.enter_async_context(stdio_client(server_params))
        self.stdio, self.write = stdio_transport
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))

        await self.session.initialize()
        response = await self.session.list_tools()
        return response
    

    def process_query(self, query: str) -> str:
        """Process a query using LLM and the available tools."""
        if not self.session:
            return "Error: Not connected to a server. Please run connect_to_server first."

        pdb.set_trace()
        messages = [{"role": "user", "content": query}]
        available_tools = [{
            "name": tool.name,
            "description": tool.description,
            "input_schema": tool.inputSchema
        } for server in self.tools.values() for tool in server]
        print(f"Sending query to {MODEL}...")
        response = self.anthropic.messages.create(
            model=MODEL, 
            max_tokens=1024,
            messages=messages,
            tools=available_tools
        )

        final_text = []
        for content in response.content:
            if content.type == 'text':
                final_text.append(content.text)
            elif content.type == 'tool_use':
                tool_name = content.name
                tool_args = content.input
                print(f"{MODEL} requested to use tool: {tool_name} with arguments: {tool_args}")

                result = asyncio.run(self.session.call_tool(tool_name, tool_args))
                print("Received tool result from server.")

                # Create the tool result content block
                tool_result_content = {
                    "type": "tool_result",
                    "tool_use_id": content.id,
                    "content": str(result.content) # Ensure content is a string
                }

                # Append the original assistant message and the tool result
                messages.append({"role": "assistant", "content": response.content})
                messages.append({"role": "user", "content": [tool_result_content]})

                # Get next response from LLM
                print(f"Tool result: {tool_result_content}")
                print(f"Sending tool result back to {MODEL}...")
                follow_up_response = self.anthropic.messages.create(
                    model=MODEL,
                    max_tokens=1024,
                    messages=messages,
                )
                for follow_up_content in follow_up_response.content:
                    if follow_up_content.type == 'text':
                        final_text.append(follow_up_content.text)

        return "\n".join(final_text)
    
    def chat_loop(self):
        """Run an interactive chat loop"""
        print("\nMCP Client Started!")
        print("Type your queries or 'quit' to exit.")

        while True:
            try:
                query = input("\nQuery: ").strip()

                if query.lower() == 'quit':
                    break

                response = self.process_query(query)
                print("\n" + response)

            except Exception as e:
                print(f"\nError: {str(e)}")
                
    def cleanup(self):
        """Clean up resources and close the server connection."""
        print("Cleaning up resources...")
        asyncio.run(self.exit_stack.aclose())
        print("Cleanup complete.")


In [13]:
def connect():
    client = MCPClient()
    client.connect_to_server('swallow_server.py')
    return client

# Run the connection and keep the client object
# This will block until the connection is established.
client = connect()


Connecting to server: swallow_server.py...
> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_22161/769315661.py(28)connect_to_server()
     26         pdb.set_trace()
     27         # print(server_params)
---> 28         response = asyncio.run(self.async_connect_to_server(server_params))
     29         # print(response)
     30         self.tools[server_script_path] = response.tools

ipdb> c

Connection successful!
Available tools: ['unladen_swallow_airspeed']


In [14]:
def run_query(query):
    response = client.process_query(query)
    print("\n--- LLM's Response ---")
    print(response)
    print("-------------------------")


In [15]:
# Run a query
query = "What is the airspeed velocity of an unladen European swallow"
run_query(query)


> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_22161/769315661.py(53)process_query()
     51 
     52         pdb.set_trace()
---> 53         messages = [{"role": "user", "content": query}]
     54         available_tools = [{
     55             "name": tool.name,

ipdb> c
Sending query to claude-sonnet-4-20250514...
claude-sonnet-4-20250514 requested to use tool: unladen_swallow_airspeed with arguments: {'swallow_type': 'european'}
Received tool result from server.
Tool result: {'type': 'tool_result', 'tool_use_id': 'toolu_01RyDUoDvJdhX1vdx3ps8xaL', 'content': "[TextContent(type='text', text='27.1828km/h', annotations=None, meta=None)]"}
Sending tool result back to claude-sonnet-4-20250514...

--- LLM's Response ---
The airspeed velocity of an unladen European swallow is approximately 27.1828 km/h.

Though I should note that this is a reference to the classic question from "Monty Python and the Holy Grail"! In reality, European swallows (barn swallows) typically fly at 

In [16]:
client.chat_loop()



MCP Client Started!
Type your queries or 'quit' to exit.

Query: what is the airspeed velocity of an unladen swallow?
> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_22161/769315661.py(53)process_query()
     51 
     52         pdb.set_trace()
---> 53         messages = [{"role": "user", "content": query}]
     54         available_tools = [{
     55             "name": tool.name,

ipdb> c
Sending query to claude-sonnet-4-20250514...

I can help you with that! However, I need to know which type of swallow you're asking about, since the airspeed velocity differs between species.

Are you asking about an African or European swallow?

Query: African
> /var/folders/6d/3xz907yn5ylg43s2vlnnzptr0000gn/T/ipykernel_22161/769315661.py(53)process_query()
     51 
     52         pdb.set_trace()
---> 53         messages = [{"role": "user", "content": query}]
     54         available_tools = [{
     55             "name": tool.name,

ipdb> c
Sending query to claude-sonnet-4-20250514